In [1]:
import numpy as np
import matplotlib.pyplot as plt
import qrisp

%load_ext autoreload

%autoreload 2

# Elliptic curve

In [2]:
class EllCurve:
    # Elliptic curve class
    def __init__(self, a, b, p):
        self.a = a
        self.b = b
        self.p = p


class EllPoint:
    # Elliptic curve point class
    def __init__(self, x, y):
        self.x = x
        self.y = y

# Neutral element       
EllZero = EllPoint(0, 0)

In [3]:
p = 7
a = 5
b = 4


# Function to compute y given x modulo p
def compute_y(x, a, b, p):
    for y in range(p):
        if (y**2 % p) == ((x**3 + a * x + b) % p):
            return y
    return None

In [4]:
def ell_add_generic(P, Q, curve):
    # return the result of P + Q
    p = curve.p
    s = (((P.y - Q.y) % p) * pow((P.x - Q.x) % p, -1, p)) % p
    xr = p + (s * s - P.x - Q.x) % p
    yr = (P.y - s * ((P.x - xr) % p)) % p
    return EllPoint((p + xr) % p, (p - yr) % p)


def ell_double(P, curve):
    p = curve.p

    s = ((3 * (P.x * P.x % p) + curve.a) % p) * pow((2 * P.y) % p, -1, p)

    xr = (s * s - 2 * P.x) % p
    yr = P.y - s * ((P.x - xr) % p) % p
    # print(xr, yr)
    return EllPoint((p + xr) % p, (p - yr) % p)


def ell_add(P, Q, curve):
    p = curve.p
    if P.x == EllZero.x and P.y == EllZero.y:
        return Q
    if Q.x == EllZero.x and Q.y == EllZero.y:
        return P
    if P.x == Q.x:
        if P.y == (p - Q.y) % p:
            return EllZero
        if P.y == Q.y and Q.y != 0:
            return ell_double(P, curve)
    return ell_add_generic(P, Q, curve)


def ell_mult_add(P, Q, k, curve):
    n = k.bit_length()
    res = Q
    power = P
    for _ in range(n):
        if k % 2:
            res = ell_add(res, power, curve)
        k >>= 1
        power = ell_double(power, curve)
    return res

In [5]:
P = EllPoint(3, 2)
points = [P]
curve = EllCurve(a, b, p)
for _ in range(10):
    points.append(ell_add(points[-1], P, curve))

for i, point in enumerate(points):
    print(f"Point {chr(65 + i)}:", point.x, point.y)

Point A: 3 2
Point B: 2 6
Point C: 4 2
Point D: 0 5
Point E: 5 0
Point F: 0 2
Point G: 4 5
Point H: 2 1
Point I: 3 5
Point J: 0 0
Point K: 3 2


In [6]:
def compute_logarithm(G, P):
    temp = G

    for i in range(1, 2*p):
        if temp.x == P.x and temp.y == P.y:
            return i
        temp = ell_add(temp, G, curve)

    return None

In [7]:
G = EllPoint(3, 2)
P = EllPoint(2, 1)
print(compute_logarithm(G, P))

8


# Quantum Elliptic Curve

In [8]:
def to_montgomery(x, p):
    n = p.bit_length()
    x *= 2**n % p
    return x

def to_standard(x, p):
    n = p.bit_length()
    x *= pow(2**n, -1, p) % p
    return x

def to_montgomery_qm(x, montgomery_shift):
    x *= pow(2, montgomery_shift, x.modulus)
    x.m = montgomery_shift

def to_standard_qm(x):
    montgomery_shift = x.m
    x*= pow(2, -montgomery_shift, x.modulus)
    x.m = 0

In [9]:
def inpl_rsub(r, p):
    qrisp.x(r)
    r.inpl_adder(r.modulus+1, r)
    r += p

In [10]:
def kaliski_quantum(v, p, m):
    n = p.bit_length()
    # Convert to Montgomery
    to_montgomery(v, p)
    u = qrisp.QuantumFloat(n)
    u[:] = p
    r = qrisp.QuantumModulus(2 * p)
    r[:] = 0
    s = qrisp.QuantumModulus(2 * p)
    s[:] = 1

    v.__class__ = qrisp.QuantumFloat

    a = qrisp.QuantumBool()
    b = qrisp.QuantumBool()
    add = qrisp.QuantumBool()
    f = qrisp.QuantumBool()
    f[:] = True
    for i in range(2 * n):
        is_zero = v == 0
        qrisp.mcx([f, is_zero], m[i])
        is_zero.uncompute()
        qrisp.cx(m[i], f)
        # STEP 1
        qrisp.mcx([f, u[0]], a, ctrl_state="10")
        qrisp.mcx([f, a, v[0]], m[i], ctrl_state="100")
        qrisp.cx(a, b)
        qrisp.cx(m[i], b)

        # STEP 2
        l = u > v
        qrisp.mcx([f, l, b], a, ctrl_state="110")
        qrisp.mcx([f, l, b], m[i], ctrl_state="110")
        l.uncompute()

        # STEP 3
        with qrisp.control(a):
            qrisp.swap(u, v)
            qrisp.swap(r, s)

        # STEP 4
        qrisp.mcx([f, b], add, ctrl_state="10")
        with qrisp.control(add):
            v -= u
            s += r
        # STEP 5
        qrisp.mcx([f, b], add, ctrl_state="10")
        # uncompute b
        qrisp.cx(m[i], b)
        qrisp.cx(a, b)

        # Division by 2
        with qrisp.control(f):
            with qrisp.invert():
                qrisp.cyclic_shift(v)

        qrisp.cyclic_shift(r)
        larger = r > p
        with larger:
            r -= p
        qrisp.cx(r[0], larger)
        larger.delete()

        with qrisp.control(a):
            qrisp.swap(u, v)
            qrisp.swap(r, s)
        # uncompute a
        qrisp.mcx([s[0]], a, ctrl_state="0")
    
    a.delete()
    add.delete()
    b.delete()

    inpl_rsub(r, p)

    v.__class__ = qrisp.QuantumModulus
    for i in range(v.size):
        qrisp.swap(v[i], r[i])

    # Uncompute u,s,f
    f.delete()
    qrisp.x(u[0])
    u.delete()
    r.delete()
    s -= p
    s.delete()
    # Convert back to standard representation
    to_standard(v, p)
    return v

In [11]:
def qrisp_ell_double(P, curve):
    p = curve.p

    s = ((3 * (P[0] * P[0] % p) + curve.a) % p) * pow((2 * P[1]) % p, -1, p)

    xr = (s * s - 2 * P[0]) % p
    yr = P[1] - s * ((P[0] - xr) % p) % p
    #CHOOSE APPROPIATE RETURN TYPE
    return [xr, (p-yr) % p]

In [12]:
# def ell_add_inpl(P, G):
#     print("P step 0:", P.x, P.y)
#     print("G", G.x, G.y)
#     P.y -= G.y
#     P.x -= G.x
#     P.x = P.x % p
#     P.y = P.y % p
#     print("P step 1-2:", P.x, P.y)
#     t0 = pow(P.x, -1, p)
#     lambda_ = (P.y * t0) % p
#     print("lstep 3:", lambda_)
#     P.y -= lambda_ * P.x
#     P.y = P.y % p
#     print("P step 4:", P.x, P.y)
#     P.x += 3*G.x
#     P.x = P.x % p
#     print("P step 5:", P.x, P.y)
#     return P

In [13]:
@qrisp.custom_control
def qrisp_ell_add_inpl(anc, G, ctrl=None):
    # return the result of P + Q
    # Following C3 in the paper
    if ctrl is None:
        anc[1] -= G[1]
    else:
        with qrisp.control(ctrl):
            anc[1] -= G[1] #step 2 //ctrl_sub_const_modp
    anc[0] -= G[0] #step 1
  
    m = qrisp.QuantumArray(qtype=qrisp.QuantumBool(), shape=(2 * p.bit_length(),))
    l = qrisp.QuantumModulus(p)
    with qrisp.conjugate(kaliski_quantum)(anc[0], p, m) as inv:
        temp = anc[1] * inv
        to_standard_qm(temp)
        l[:] = temp#step 3 & 4 & 6
        temp.uncompute()
    for a in m:
        a.delete()
    #step 5
    temp = l*anc[0]
    to_standard_qm(temp)
    anc[1]-= temp
    temp.uncompute()
    #anc[1].delete()

    if ctrl is None:
        anc[0] += 3*G[0]
    else:
        with qrisp.control(ctrl):
            anc[0] += 3*G[0] #step 9 //ctrl_add_const_modp

    temp = l*l #step 7
    to_standard_qm(temp)
    if ctrl is None:
        anc[0] -= temp #step 8 
    else:
        with qrisp.control(ctrl):
            anc[0] -= temp #step 8 //ctrl_sub_modp
    temp.uncompute() #step 10

    #step 11
    temp = l*anc[0]
    to_standard_qm(temp)
    anc[1]+= temp
    temp.uncompute()

    
    m = qrisp.QuantumArray(qtype=qrisp.QuantumBool(), shape=(2 * p.bit_length(),))

    with qrisp.conjugate(kaliski_quantum)(anc[0], p, m) as inv:
        temp = anc[1] * inv
        to_standard_qm(temp)
        qrisp.cx(temp, l)
        temp.uncompute()
        
    for a in m:
        a.delete()
    
    l.delete()
    if ctrl is None:
        anc[1] -= G[1] #step 16
    else:
        with qrisp.control(ctrl):
            anc[1] -= G[1] #step 16 //ctrl_sub_const_modp

    anc[0] -= G[0] #step 17

    if ctrl is None:
        inpl_rsub(anc[0], p) #step 15
    else:
        with qrisp.control(ctrl):
            inpl_rsub(anc[0], p) #step 15 //ctrl_neg_modp

    return anc

In [14]:
def qrisp_ell_mult_add(power, res, k, curve):
    #Elliptic curve multiplication Q + kP
    n = k.size

    qrisp.merge([res, k])
    with qrisp.IterationEnvironment(res.qs, n, precompile = True):
        with qrisp.control(k[0]):
            res = qrisp_ell_add_inpl(res, power)
        with qrisp.invert():
            qrisp.cyclic_shift(k)
        power = qrisp_ell_double(power,curve)
    return res

### TEST 

#### Testing the doubling function

In [16]:
p=7
a=5
b=4
curve = EllCurve(a, b, p)
# Sample input points
input_points = [(0,2),(0,5),(2,1),(2,6),(3,2),(3,5),(4,2),(4,5)] 

# Compare outputs
for point in input_points:
    P_ell = EllPoint(point[0], point[1])
    P_qrisp = point
    
    result_ell = ell_double(P_ell, curve)
    result_qrisp = qrisp_ell_double(P_qrisp, curve)
    
    print("Input Point:", point)
    print("Elliptic Double:", (result_ell.x, result_ell.y))
    print("QRISP Double:", tuple(result_qrisp))
    print()


Input Point: (0, 2)
Elliptic Double: (2, 6)
QRISP Double: (2, 6)

Input Point: (0, 5)
Elliptic Double: (2, 1)
QRISP Double: (2, 1)

Input Point: (2, 1)
Elliptic Double: (0, 2)
QRISP Double: (0, 2)

Input Point: (2, 6)
Elliptic Double: (0, 5)
QRISP Double: (0, 5)

Input Point: (3, 2)
Elliptic Double: (2, 6)
QRISP Double: (2, 6)

Input Point: (3, 5)
Elliptic Double: (2, 1)
QRISP Double: (2, 1)

Input Point: (4, 2)
Elliptic Double: (0, 2)
QRISP Double: (0, 2)

Input Point: (4, 5)
Elliptic Double: (0, 5)
QRISP Double: (0, 5)



#### Testing the addition

In [18]:
p=7
a=5
b=4
curve = EllCurve(a, b, p)
# Sample input points
input_points = [(2,1), (4,5), (0,2), (5,0), (0,5), (4,2), (2,6) ] 
G_ell = EllPoint(3, 5)
G_qrisp = [3, 5]
mod_p = qrisp.QuantumModulus(p)

# Compare outputs
for point in input_points:
    P_ell = EllPoint(point[0], point[1])
    anc = qrisp.QuantumArray(qtype=mod_p, shape=(2,))
    #initialize the store register to the point P_0 
    anc[:] = [point[0], point[1]]
    result_ell = ell_add_generic(P_ell, G_ell, curve)
    result_qrisp = qrisp_ell_add_inpl(anc, G_qrisp)    
    print("Input Point:", point)
    print("Elliptic addition:", (result_ell.x, result_ell.y))
    print(qrisp.multi_measurement(result_qrisp))
    print()

Input Point: (2, 1)
Elliptic addition: (4, 5)
{(4, 5): 1.0}                                                                        

Input Point: (4, 5)
Elliptic addition: (0, 2)
{(0, 2): 1.0}                                                                        

Input Point: (0, 2)
Elliptic addition: (5, 0)


### Profiling

In [15]:
p=7
a=5
b=4

x1 = qrisp.QuantumModulus(2**(p.bit_length())) 
x1[:] = 1
mod_p = qrisp.QuantumModulus(p)
#store registers
anc = qrisp.QuantumArray(qtype=mod_p, shape=(2,))
#initialize the store register to the point P_0 = (3,2)
anc[:] = [2,6]

#Generator G = [2,2]
G = [0,5]

res = qrisp_ell_mult_add(G,anc,x1,curve)
len([qb for qb in res[0].qs.qubits if qb.allocated])

9

In [ ]:
qrisp.multi_measurement(res)

In [15]:
import cProfile
with cProfile.Profile() as pr:
    p=7
    a=5
    b=4

    x1 = qrisp.QuantumModulus(2**(p.bit_length())) 
    x1[:] = 1
    mod_p = qrisp.QuantumModulus(p)
    #store registers
    anc = qrisp.QuantumArray(qtype=mod_p, shape=(2,))
    #initialize the store register to the point P_0 = (3,2)
    anc[:] = [2,6]

    #Generator G = [2,2]
    G = [0,5]

    res = qrisp_ell_mult_add(G,anc,x1,curve)
    #res.qs.compile()
    print(qrisp.multi_measurement(res))

Simulating 37 qubits.. |██████████████████████▉                              | [ 43%]

In [ ]:
import pstats
results = pstats.Stats(pr).sort_stats(pstats.SortKey.TIME)
results.print_stats()

         156095781 function calls (155485758 primitive calls) in 60.172 seconds

   Ordered by: internal time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
105824/12715    9.899    0.000   11.395    0.001 C:\Users\dpolimen\Documents\GitHub\Qrisp\src\qrisp\circuit\transpiler.py:90(transpile_inner)
      215    7.595    0.035   14.466    0.067 C:\Users\dpolimen\Documents\GitHub\Qrisp\src\qrisp\uncomputation\unqomp.py:75(dag_from_qc)
      164    2.992    0.018    2.992    0.018 C:\Users\dpolimen\Documents\GitHub\Qrisp\src\qrisp\misc\depth_reduction.py:143(depth_sensitive_topological_sort)
  1451642    2.837    0.000    3.942    0.000 c:\Users\dpolimen\anaconda3\envs\venvABqiskitprovider\lib\site-packages\networkx\classes\digraph.py:643(add_edge)
 40666662    2.697    0.000    2.697    0.000 C:\Users\dpolimen\Documents\GitHub\Qrisp\src\qrisp\circuit\qubit.py:67(__eq__)
      483    2.643    0.005    5.711    0.012 C:\Users\dpolimen\Documents\GitHub\Qrisp\src\qri

# Whole algorithm

In [ ]:
p=5
a=3
b=3
curve = EllCurve(a, b, p)

x1 = qrisp.QuantumModulus(2**(p.bit_length())) 
x2 = qrisp.QuantumModulus(2**(p.bit_length()))

mod_p = qrisp.QuantumModulus(p)
#store registers
anc = qrisp.QuantumArray(qtype=mod_p, shape=(2,))
#initialize the store register to the point P_0 = [4,3]
anc[:] = [3,2]

#Generator G = [3,2]
G = [3,2]

#P = (5,0)
P = [3,3]

#Superposition
qrisp.h(x1)
qrisp.h(x2)
#Multiplication step: anc = P0 + x1*G 
#anc_x, anc_y = ell_mult_add_q(P,anc,x1,curve)
anc = qrisp_ell_mult_add(G, anc, x1, curve)

#Multiplication step: anc = anc - x2*P
anc = qrisp_ell_mult_add(P, anc, x2, curve)

#Quantum Fourier Transform on x1 and x2 registers
qrisp.QFT(x1, inv=True)
qrisp.QFT(x2, inv=True)

<QuantumModulus 'qf_33'>

In [ ]:
print(len([qb for qb in anc[0].qs.qubits if qb.allocated]))
print(anc[0].qs.depth())

12
571428


In [ ]:
qrisp.multi_measurement(anc)

https://graui.de/code/elliptic2/ 